## Lấy dữ liệu từ app cụ thể là từ cơ sở dữ liệu firebase

In [0]:
# Cài đặt thư viện kết nối Firebase
%pip install firebase-admin

In [0]:
import firebase_admin
from firebase_admin import credentials, firestore
import pandas as pd

print("Đang kết nối với Firebase Cloud Firestore...")

# 1. Trỏ đường dẫn tới file chìa khóa JSON bạn vừa tải lên Volume
key_path = "/Volumes/workspace/default/firebase_key/firebase_key.json"
cred = credentials.Certificate(key_path)

# Tránh lỗi khởi tạo lại kết nối nhiều lần trên Databricks
if not firebase_admin._apps:
    firebase_admin.initialize_app(cred)

db = firestore.client()

# 2. KÉO DỮ LIỆU VỀ (Extract)
collection_name = 'users' 
docs = db.collection(collection_name).stream()

data = []
for doc in docs:
    row = doc.to_dict() # Lấy toàn bộ dữ liệu trong document
    row['doc_id'] = doc.id # Lấy luôn ID để dễ quản lý
    data.append(row)

# 3. Đưa dữ liệu vào Tầng Bronze (Dữ liệu thô)
if len(data) > 0:
    raw_df = pd.DataFrame(data)
    # Chuyển Pandas DataFrame sang Spark DataFrame để dùng sức mạnh Big Data
    spark_raw_df = spark.createDataFrame(raw_df)
    
    print(f"✅ Rút dữ liệu thành công! Có {len(data)} dòng mới từ Firebase.")
    display(spark_raw_df)
else:
    print("⚠️ Không có dữ liệu nào trong Collection này!")

%md
## Lấy dữ liệu từ app, làm sạch chúng và phân tích dữ liệu và gửi cho admin để nắm bắt và phân tích tình hình của app

### đầu tiên chúng ta số lượng user rất ít nên data hiếm nên chúng ta sẽ tự giả lập để sinh ra data để làm việc này

In [0]:
import pandas as pd
import random
from datetime import datetime, timedelta

print("Đang khởi tạo 1.000 dòng dữ liệu giả lập từ App Mắt Thần...")

# 1. Định nghĩa các dữ liệu ngẫu nhiên
users = [f"User_{i}" for i in range(1, 11)]
objects = ['pothole', 'dog', 'car', 'motorcycle', 'tree_branch', 'stairs']

data = []
for _ in range(1000):
    # Giả lập tọa độ quanh khu vực TP.HCM
    lat = 10.8231 + random.uniform(-0.05, 0.05)
    lng = 106.6297 + random.uniform(-0.05, 0.05)
    
    data.append({
        'user_id': random.choice(users),
        'timestamp': (datetime.now() - timedelta(days=random.randint(0, 30))).strftime('%Y-%m-%d %H:%M:%S'),
        'latitude': round(lat, 6),
        'longitude': round(lng, 6),
        'detected_object': random.choice(objects),
        'confidence_score': round(random.uniform(0.4, 0.99), 2)
    })

df = pd.DataFrame(data)

# 2. Cố tình tạo ra dữ liệu "Bẩn" (Dirty Data) để lát nữa thực hành làm sạch (Silver layer)
# Xóa trắng tọa độ của 20 dòng ngẫu nhiên (Giả lập việc điện thoại mất sóng GPS)
df.loc[10:30, ['latitude', 'longitude']] = None

# 3. Lưu thành file CSV trực tiếp vào Volume của bạn
csv_path = '/Volumes/workspace/default/dataset_ổ_gà/app_logs_raw.csv'
df.to_csv(csv_path, index=False)

print(f"Xong! Đã lưu file dữ liệu thô vào: {csv_path}")
print("Xem thử 5 dòng đầu tiên:")
display(df.head())

## sau khi đã có file data .csv thì chúng ta sẽ làm sạch chúng 

In [0]:
from pyspark.sql.functions import col, to_timestamp

# 1. Đọc dữ liệu từ file CSV thô (Tầng Bronze)
raw_path = "/Volumes/workspace/default/dataset_ổ_gà/app_logs_raw.csv"
raw_df = spark.read.csv(raw_path, header=True, inferSchema=True)

print(f"📊 Số dòng ban đầu (Bronze): {raw_df.count()}")

# 2. Xử lý và làm sạch (Tầng Silver)
# Lọc bỏ các dòng bị rớt mạng GPS (latitude hoặc longitude bị rỗng)
clean_df = raw_df.dropna(subset=["latitude", "longitude"])

# Chuẩn hóa cột thời gian
clean_df = clean_df.withColumn("timestamp", to_timestamp(col("timestamp"), "yyyy-MM-dd HH:mm:ss"))

print(f"✨ Số dòng sau khi dọn rác (Silver): {clean_df.count()}")

# 3. Xuất file dữ liệu sạch
clean_csv_path = "/Volumes/workspace/default/dataset_ổ_gà/app_logs_clean"
# Dùng coalesce(1) để gom data về 1 file CSV duy nhất cho dễ quản lý
clean_df.coalesce(1).write.mode("overwrite").csv(clean_csv_path, header=True)

print(f"✅ Đã lưu dữ liệu sạch thành định dạng CSV tại: {clean_csv_path}")
display(clean_df.head(5))

## sau khi làm sạch chúng ta sẽ làm phần nghiệp vụ đó là đếm xem vật cản nào dang gây nguy hiểm cao cho người dùng và tổng hợp chúng lại

In [0]:
from pyspark.sql.functions import desc

# Gom nhóm theo tên vật thể và đếm số lượng
gold_df = clean_df.groupBy("detected_object") \
                  .count() \
                  .withColumnRenamed("count", "total_detected") \
                  .orderBy(desc("total_detected"))

print("🏆 BẢNG TỔNG HỢP VẬT CẢN NGUY HIỂM NHẤT (Tầng Gold):")
display(gold_df)

## tiếp theo chúng ta sẽ phân tích các địa điểm mà mật độ có vật thể nguy hiểm dày dặt nhất để ta có thể cảnh báo sớm cho user sớm rằng những địa điểm bạn sắp tới sẽ có nhiều vật cản nguy hiểm nên cẩn thận hơn. Ta sẽ sử dụng thuật toán gom cụm K-Mean để gom các vật thể nguy hiểm thành 1 cụm và phân tích xem chỗ nào nhiều chỗ nào ít

In [0]:
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
from pyspark.sql.functions import col

# 1. Chuẩn bị dữ liệu: Gom tọa độ Lat/Lng thành một vector đặc trưng
# Chúng ta lấy dữ liệu từ clean_df (Tầng Silver đã làm ở bước trước)
vec_assembler = VectorAssembler(inputCols=["latitude", "longitude"], outputCol="features")
final_data = vec_assembler.transform(clean_df)

# 2. Khởi tạo thuật toán K-Means
# k=5 nghĩa là chúng ta muốn AI tìm ra 5 cụm (điểm đen) lớn nhất
kmeans = KMeans(featuresCol="features", k=5, seed=42)

# 3. Huấn luyện mô hình
model = kmeans.fit(final_data)

# 4. Xác định tọa độ tâm của các "Điểm đen"
centers = model.clusterCenters()

print("📍 TỌA ĐỘ 5 ĐIỂM ĐEN NGUY HIỂM NHẤT ĐÃ ĐƯỢC XÁC ĐỊNH:")
blackspots_data = []
for i, center in enumerate(centers):
    print(f"Điểm đen {i+1}: Vĩ độ {center[0]:.6f}, Kinh độ {center[1]:.6f}")
    blackspots_data.append((i+1, float(center[0]), float(center[1])))

# 5. Tạo bảng kết quả để lưu vào Tầng Gold
blackspots_df = spark.createDataFrame(blackspots_data, ["id", "latitude", "longitude"])

# Lưu kết quả xuống Volume để App hoặc Dashboard có thể truy vấn
gold_path = "/Volumes/workspace/default/dataset_ổ_gà/blackspots_gold"
blackspots_df.write.mode("overwrite").parquet(gold_path)

print(f"\n✅ Đã lưu tọa độ điểm đen vào Tầng Gold: {gold_path}")
display(blackspots_df)

Databricks visualization. Run in Databricks to view.